<a href="https://colab.research.google.com/github/sujith2359/Gen_AI_Assessments/blob/main/Text_Chunking_and_Semantic_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Chunking and Semantic Retrieval for GenAI

## INSTALLATION AND IMPORTS

In [13]:
!pip install -q langchain-text-splitters sentence-transformers scikit-learn pandas chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [15]:
import re
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

print("Imports completed successfully!")

Imports completed successfully!


In [16]:
source_document = """Generative AI models, specifically Large Language Models (LLMs), represent a paradigm shift in human-computer interaction. These models are trained on vast datasets to predict the next word, allowing them to write articles, debug code, and answer user queries with human-like ease. However, they suffer from a major limitation: hallucinations, where they generate plausible-sounding but completely incorrect information. This occurs because LLMs operate purely on statistical patterns, lacking a real-time connection to a verified source of facts.

To address this, Retrieval-Augmented Generation (RAG) integrates a dynamic external document database into the workflow. Instead of relying solely on parametric memory, the system uses semantic search to fetch relevant snippets of text directly mapping to the user's inquiry. Semantic search maps text to dense vector spaces using embeddings. These numerical vectors capture semantic relationships beyond keyword matching.

When a prompt is received, the RAG engine generates a query embedding and seeks nearest-neighbor documents inside a specialized vector database. The system retrieves the most informative blocks, appending them as dynamic context. Finally, the context-aware LLM generates its response based on this verified reference ground truth, significantly reducing hallucination while keeping answers accurate and timely."""

# Basic validation
if not source_document.strip():
    raise ValueError("The source document is empty!")

print(f"Document successfully created!")
print(f"Total length of source document: {len(source_document)} characters.")

Document successfully created!
Total length of source document: 1383 characters.


In [17]:
def create_fixed_chunks(text, chunk_length=250):
    if not text.strip():
        return []
    chunks = []
    for i in range(0, len(text), chunk_length):
        chunks.append(text[i:i + chunk_length])
    return chunks

chunk_length_config = 250
fixed_chunks = create_fixed_chunks(source_document, chunk_length=chunk_length_config)

print(f"Fixed-Size Chunking (Size: {chunk_length_config} characters):")
print(f"Total chunks generated: {len(fixed_chunks)}\n")

for index, segment in enumerate(fixed_chunks):
    print(f"--- Chunk {index + 1} ({len(segment)} characters) ---")
    print(segment.strip())

Fixed-Size Chunking (Size: 250 characters):
Total chunks generated: 6

--- Chunk 1 (250 characters) ---
Generative AI models, specifically Large Language Models (LLMs), represent a paradigm shift in human-computer interaction. These models are trained on vast datasets to predict the next word, allowing them to write articles, debug code, and answer use
--- Chunk 2 (250 characters) ---
r queries with human-like ease. However, they suffer from a major limitation: hallucinations, where they generate plausible-sounding but completely incorrect information. This occurs because LLMs operate purely on statistical patterns, lacking a real
--- Chunk 3 (250 characters) ---
-time connection to a verified source of facts.

To address this, Retrieval-Augmented Generation (RAG) integrates a dynamic external document database into the workflow. Instead of relying solely on parametric memory, the system uses semantic search
--- Chunk 4 (250 characters) ---
to fetch relevant snippets of text directly m

In [18]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40
)

recursive_chunks = recursive_splitter.split_text(source_document)

print(f"Recursive Character Chunking (Size: 250, Overlap: 40):")
print(f"Total chunks generated: {len(recursive_chunks)}\n")

for index, segment in enumerate(recursive_chunks):
    print(f"--- Chunk {index + 1} ({len(segment)} characters) ---")
    print(segment.strip())

Recursive Character Chunking (Size: 250, Overlap: 40):
Total chunks generated: 7

--- Chunk 1 (246 characters) ---
Generative AI models, specifically Large Language Models (LLMs), represent a paradigm shift in human-computer interaction. These models are trained on vast datasets to predict the next word, allowing them to write articles, debug code, and answer
--- Chunk 2 (245 characters) ---
write articles, debug code, and answer user queries with human-like ease. However, they suffer from a major limitation: hallucinations, where they generate plausible-sounding but completely incorrect information. This occurs because LLMs operate
--- Chunk 3 (126 characters) ---
This occurs because LLMs operate purely on statistical patterns, lacking a real-time connection to a verified source of facts.
--- Chunk 4 (244 characters) ---
To address this, Retrieval-Augmented Generation (RAG) integrates a dynamic external document database into the workflow. Instead of relying solely on parametric memor

In [19]:
char_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=300,
    chunk_overlap=20
)

char_chunks = char_splitter.split_text(source_document)

print(f"Character-Based Chunking (Separator: '\\n', Size: 300, Overlap: 20):")
print(f"Total chunks generated: {len(char_chunks)}\n")

for index, segment in enumerate(char_chunks):
    print(f"--- Chunk {index + 1} ({len(segment)} characters) ---")
    print(segment.strip())

Character-Based Chunking (Separator: '\n', Size: 300, Overlap: 20):
Total chunks generated: 3

--- Chunk 1 (547 characters) ---
Generative AI models, specifically Large Language Models (LLMs), represent a paradigm shift in human-computer interaction. These models are trained on vast datasets to predict the next word, allowing them to write articles, debug code, and answer user queries with human-like ease. However, they suffer from a major limitation: hallucinations, where they generate plausible-sounding but completely incorrect information. This occurs because LLMs operate purely on statistical patterns, lacking a real-time connection to a verified source of facts.
--- Chunk 2 (422 characters) ---
To address this, Retrieval-Augmented Generation (RAG) integrates a dynamic external document database into the workflow. Instead of relying solely on parametric memory, the system uses semantic search to fetch relevant snippets of text directly mapping to the user's inquiry. Semantic search

In [20]:
def make_sentence_chunks(text, sentences_per_group=3):
    # Simple regex to split sentences keeping punctuation
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if s]

    if not sentences:
        return []

    chunks = []
    for i in range(0, len(sentences), sentences_per_group):
        group = sentences[i:i + sentences_per_group]
        chunks.append(" ".join(group))
    return chunks

sentence_chunks = make_sentence_chunks(source_document, sentences_per_group=3)

print(f"Sentence-Based Chunking (Sentences per group: 3):")
print(f"Total chunks generated: {len(sentence_chunks)}\n")

for index, segment in enumerate(sentence_chunks):
    print(f"--- Chunk {index + 1} ({len(segment)} characters) ---")
    print(segment.strip())

Sentence-Based Chunking (Sentences per group: 3):
Total chunks generated: 4

--- Chunk 1 (420 characters) ---
Generative AI models, specifically Large Language Models (LLMs), represent a paradigm shift in human-computer interaction. These models are trained on vast datasets to predict the next word, allowing them to write articles, debug code, and answer user queries with human-like ease. However, they suffer from a major limitation: hallucinations, where they generate plausible-sounding but completely incorrect information.
--- Chunk 2 (402 characters) ---
This occurs because LLMs operate purely on statistical patterns, lacking a real-time connection to a verified source of facts. To address this, Retrieval-Augmented Generation (RAG) integrates a dynamic external document database into the workflow. Instead of relying solely on parametric memory, the system uses semantic search to fetch relevant snippets of text directly mapping to the user's inquiry.
--- Chunk 3 (291 characters) ---


In [21]:
strategies = {
    "Fixed-size": fixed_chunks,
    "Character-based": char_chunks,
    "Recursive character-based": recursive_chunks,
    "Sentence-based": sentence_chunks
}

comparison_records = []

for name, list_of_chunks in strategies.items():
    if len(list_of_chunks) == 0:
        lengths = [0]
    else:
        lengths = [len(c) for c in list_of_chunks]

    comparison_records.append({
        "Strategy": name,
        "Number of Chunks": len(list_of_chunks),
        "Average Characters": round(np.mean(lengths), 1),
        "Minimum Characters": np.min(lengths),
        "Maximum Characters": np.max(lengths)
    })

df_comparison = pd.DataFrame(comparison_records)
display(df_comparison)

,Strategy,Number of Chunks,Average Characters,Minimum Characters,Maximum Characters
0,Fixed-size,6,230.5,133,250
1,Character-based,3,459.7,410,547
2,Recursive character-based,7,217.1,126,246
3,Sentence-based,4,344.5,265,420


In [22]:
test_sizes = [250, 400, 600]
size_comparison = []

for size in test_sizes:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=0)
    current_chunks = splitter.split_text(source_document)

    lengths = [len(c) for c in current_chunks] if current_chunks else [0]

    size_comparison.append({
        "Configured Chunk Size": size,
        "Number of Chunks": len(current_chunks),
        "Average Chunk Size": round(np.mean(lengths), 1)
    })

df_sizes = pd.DataFrame(size_comparison)
display(df_sizes)

print("\n--- Key Observations ---")
print("1. As configured chunk size increases, the total number of generated chunks decreases.")
print("2. Larger chunk sizes capture wider contextual blocks but can risk merging unrelated semantic statements.")

,Configured Chunk Size,Number of Chunks,Average Chunk Size
0,250,7,196.4
1,400,6,229.3
2,600,3,459.7



--- Key Observations ---
1. As configured chunk size increases, the total number of generated chunks decreases.
2. Larger chunk sizes capture wider contextual blocks but can risk merging unrelated semantic statements.


In [23]:
# Load the transformer model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create 5 original sentences
sentences = [
    "Generative models synthesize original text based on learned patterns.",
    "Retrieval-Augmented Generation integrates external documents to minimize hallucination.",
    "A vector database indexes numeric representation matrices for quick lookup.",
    "Large Language Models process prompt contexts to predict the next logical outputs.",
    "Semantic search allows systems to understand intent beyond literal keyword queries."
]

# Generate the numerical embeddings
embeddings = embed_model.encode(sentences)

print(f"Total sentences encoded: {len(sentences)}")
print(f"Dimensionality of embeddings: {embeddings.shape[1]}")
print(f"First few elements of Sentence 1 embedding:\n{embeddings[0][:6]}...")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total sentences encoded: 5
Dimensionality of embeddings: 384
First few elements of Sentence 1 embedding:
[-0.10050184 -0.03781808  0.03392869  0.06709215 -0.02566004  0.00832354]...


In [24]:
# 1. Related comparison (RAG vs LLMs)
sim_related = cosine_similarity([embeddings[1]], [embeddings[3]])[0][0]

# 2. Unrelated comparison
unrelated_sent_1 = "Space exploration requires rigorous rocketry calculations and physical planning."
unrelated_sent_2 = "Pancakes are best served with sweet maple syrup and fresh blueberries."

unrelated_embed_1 = embed_model.encode([unrelated_sent_1])
unrelated_embed_2 = embed_model.encode([unrelated_sent_2])

sim_unrelated = cosine_similarity(unrelated_embed_1, unrelated_embed_2)[0][0]

print("=== SIMILARITY REPORT ===\n")
print(f"Sentence A: '{sentences[1]}'")
print(f"Sentence B: '{sentences[3]}'")
print(f"Cosine Similarity (Related): {sim_related:.4f}\n")

print(f"Sentence Y: '{unrelated_sent_1}'")
print(f"Sentence Z: '{unrelated_sent_2}'")
print(f"Cosine Similarity (Unrelated): {sim_unrelated:.4f}\n")

print("--- Explanatory Conclusion ---")
print("Semantically related sentences obtain higher cosine similarity because their neural representations share direction vector space.")
print("These representations capture mathematical patterns derived from semantic contexts, ensuring synonyms align tightly.")

=== SIMILARITY REPORT ===

Sentence A: 'Retrieval-Augmented Generation integrates external documents to minimize hallucination.'
Sentence B: 'Large Language Models process prompt contexts to predict the next logical outputs.'
Cosine Similarity (Related): 0.2230

Sentence Y: 'Space exploration requires rigorous rocketry calculations and physical planning.'
Sentence Z: 'Pancakes are best served with sweet maple syrup and fresh blueberries.'
Cosine Similarity (Unrelated): -0.0589

--- Explanatory Conclusion ---
Semantically related sentences obtain higher cosine similarity because their neural representations share direction vector space.
These representations capture mathematical patterns derived from semantic contexts, ensuring synonyms align tightly.


In [25]:
documents_pool = [
    "Large Language Models undergo intensive training to construct massive parametric weights.",
    "A custom knowledge base serves as a factual anchor to verify incoming prompts.",
    "RAG pipelines leverage prompt engineering and document loaders to assemble final context.",
    "Vector databases are optimized to store, index, and query structural embeddings efficiently.",
    "Semantic distance calculates coordinate alignment using cosine similarity metrics.",
    "Language modeling allows generative platforms to draft conversational emails seamlessly."
]

# Generate index embeddings
doc_embeddings = embed_model.encode(documents_pool)

# User Query
search_query = "How does retrieval help a language model answer questions?"
query_embedding = embed_model.encode([search_query])

# Calculate similarities
similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

# Pack results into DataFrame
search_results = pd.DataFrame({
    "Document ID": [f"Doc_{i}" for i in range(len(documents_pool))],
    "Document": documents_pool,
    "Similarity Score": similarities
})

# Sort and show top 3 results
top_3_results = search_results.sort_values(by="Similarity Score", ascending=False).head(3)
display(top_3_results)

,Document ID,Document,Similarity Score
0,Doc_0,Large Language Models undergo intensive traini...,0.404899
5,Doc_5,Language modeling allows generative platforms ...,0.402372
4,Doc_4,Semantic distance calculates coordinate alignm...,0.278135


In [26]:
# Initialize an ephemeral client (in-memory, clean slate)
chroma_client = chromadb.Client()

collection_name = "genai_knowledge_repository"

# Delete the collection first if it exists to ensure idempotent runs
try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(name=collection_name)

# Prepare documents, metadata and IDs
chroma_docs = [
    "Retrieval-Augmented Generation (RAG) decreases generation hallucination.",
    "Sentence transformers convert strings into semantic embedding vectors.",
    "Generative LLMs excel at language synthesis and contextual translation.",
    "Vector indexing speeds up search inside extremely large databases.",
    "Cosine similarity metrics quantify relative orientation between vectors.",
    "Hybrid semantic pipelines fuse search engines with generative decoders."
]

chroma_metadatas = [
    {"category": "RAG"},
    {"category": "Embedding"},
    {"category": "LLM"},
    {"category": "VectorDB"},
    {"category": "Search"},
    {"category": "RAG"}
]

chroma_ids = [f"id_{i+1}" for i in range(len(chroma_docs))]

# Add documents. Chroma will auto-compute embeddings using its default model if we don't supply custom ones.
collection.add(
    documents=chroma_docs,
    metadatas=chroma_metadatas,
    ids=chroma_ids
)

print(f"Collection created! Total document count loaded: {collection.count()}")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 70.6MiB/s]


Collection created! Total document count loaded: 6


In [27]:
query_text = "What is the purpose of embeddings in AI search systems?"

query_results = collection.query(
    query_texts=[query_text],
    n_results=3
)

print(f"Query: '{query_text}'\n")
for i in range(len(query_results['ids'][0])):
    print(f"Rank {i+1}:")
    print(f"  ID: {query_results['ids'][0][i]}")
    print(f"  Document: {query_results['documents'][0][i]}")
    print(f"  Distance: {query_results['distances'][0][i]:.4f}")
    print(f"  Metadata: {query_results['metadatas'][0][i]}\n")

Query: 'What is the purpose of embeddings in AI search systems?'

Rank 1:
  ID: id_2
  Document: Sentence transformers convert strings into semantic embedding vectors.
  Distance: 1.1735
  Metadata: {'category': 'Embedding'}

Rank 2:
  ID: id_6
  Document: Hybrid semantic pipelines fuse search engines with generative decoders.
  Distance: 1.1788
  Metadata: {'category': 'RAG'}

Rank 3:
  ID: id_4
  Document: Vector indexing speeds up search inside extremely large databases.
  Distance: 1.2441
  Metadata: {'category': 'VectorDB'}



In [28]:
filtered_results = collection.query(
    query_texts=["How can we optimize response quality?"],
    where={"category": "RAG"},
    n_results=2
)

print("Filtered Query Results (Category: RAG):\n")
for i in range(len(filtered_results['ids'][0])):
    print(f"ID: {filtered_results['ids'][0][i]}")
    print(f"Document: {filtered_results['documents'][0][i]}")
    print(f"Metadata: {filtered_results['metadatas'][0][i]}\n")

Filtered Query Results (Category: RAG):

ID: id_6
Document: Hybrid semantic pipelines fuse search engines with generative decoders.
Metadata: {'category': 'RAG'}

ID: id_1
Document: Retrieval-Augmented Generation (RAG) decreases generation hallucination.
Metadata: {'category': 'RAG'}



In [29]:
target_id = "id_3"
updated_text = "State-of-the-art Generative LLMs rely on massive multi-head self-attention mechanisms to generate human-grade language patterns."
updated_metadata = {"category": "LLM_Advanced"}

# Update
collection.update(
    ids=[target_id],
    documents=[updated_text],
    metadatas=[updated_metadata]
)

# Verify
verification_result = collection.get(ids=[target_id])
print("Successfully Updated Document:")
print(f"ID: {verification_result['ids'][0]}")
print(f"Document Content: {verification_result['documents'][0]}")
print(f"Metadata: {verification_result['metadatas'][0]}")

Successfully Updated Document:
ID: id_3
Document Content: State-of-the-art Generative LLMs rely on massive multi-head self-attention mechanisms to generate human-grade language patterns.
Metadata: {'category': 'LLM_Advanced'}


In [30]:
delete_id = "id_5"

print(f"Document count before deletion: {collection.count()}")

# Delete execution
collection.delete(ids=[delete_id])

print(f"Document count after deletion: {collection.count()}")
try:
    check_deleted = collection.get(ids=[delete_id])
    print(f"Does id_5 still exist? {'Yes' if check_deleted['ids'] else 'No'}")
except Exception as e:
    print(f"Id check returned exception: {e}")

Document count before deletion: 6
Document count after deletion: 5
Does id_5 still exist? No


In [31]:
rag_query = "How to resolve model hallucination and improve facts?"

# Retrieve contextual segments
retrieved_results = collection.query(
    query_texts=[rag_query],
    n_results=2
)

retrieved_documents = retrieved_results['documents'][0]
context_block = "\n- ".join(retrieved_documents)

# Local Python 'generator' logic summarizing retrieved context
def generate_answer_from_context(query, context_list):
    # Simple conditional mapping simulating factual assembly
    combined_raw_text = " ".join(context_list).lower()

    if "hallucination" in combined_raw_text or "rag" in combined_raw_text:
        return ("Based on retrieved references, retrieval-augmented generation systems effectively combat "
                "hallucination by grounding model responses in factual external contexts instead of speculative parametric weights.")
    else:
        return "The retrieved context does not contain sufficient details to build a verified factual response."

final_answer = generate_answer_from_context(rag_query, retrieved_documents)

print("==================================================")
print(f"USER QUERY:        {rag_query}")
print("==================================================")
print(f"RETRIEVED CONTEXT:\n- {context_block}")
print("==================================================")
print(f"FINAL ANSWER:      {final_answer}")
print("==================================================")

USER QUERY:        How to resolve model hallucination and improve facts?
RETRIEVED CONTEXT:
- Retrieval-Augmented Generation (RAG) decreases generation hallucination.
- State-of-the-art Generative LLMs rely on massive multi-head self-attention mechanisms to generate human-grade language patterns.
FINAL ANSWER:      Based on retrieved references, retrieval-augmented generation systems effectively combat hallucination by grounding model responses in factual external contexts instead of speculative parametric weights.


In [35]:
analysis_records = [
    {
        "Technique": "Recursive Splitter",
        "Purpose": "Generates overlaps dynamically avoiding cut sentences",
        "Number of Chunks / Results": len(recursive_chunks),
        "Average Size or Similarity": f"{df_comparison.loc[df_comparison['Strategy'] == 'Recursive character-based', 'Average Characters'].values[0]} chars"
    },
    {
        "Technique": "Sentence Similarity",
        "Purpose": "Calculates proximity between coordinate vector representations",
        "Number of Chunks / Results": 2,
        "Average Size or Similarity": f"{sim_related:.4f} score"
    },
    {
        "Technique": "ChromaDB Query",
        "Purpose": "Retrieves targeted database entries without linear table scans",
        "Number of Chunks / Results": 3,
        "Average Size or Similarity": f"{query_results['distances'][0][0]:.4f} distance"
    }
]

df_analysis = pd.DataFrame(analysis_records)
display(df_analysis)

print("\nText Chunking and Semantic Retrieval Experiment Completed Successfully.")

,Technique,Purpose,Number of Chunks / Results,Average Size or Similarity
0,Recursive Splitter,Generates overlaps dynamically avoiding cut se...,7,217.1 chars
1,Sentence Similarity,Calculates proximity between coordinate vector...,2,0.2230 score
2,ChromaDB Query,Retrieves targeted database entries without li...,3,1.1735 distance



Text Chunking and Semantic Retrieval Experiment Completed Successfully.
